In [54]:
from dotenv import load_dotenv
load_dotenv()

True

In [55]:
import os
os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

In [56]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

In [57]:
len(embeddings.embed_query("hello AI world"))

768

In [58]:
# Importing Pinecone client
from pinecone import Pinecone
pinecone_api_key = os.environ.get("PINECONE_API_KEY")

In [59]:
# Initialize Pinecone client
pc=Pinecone(api_key=pinecone_api_key)

In [60]:
# Check if the index exists, create it if not
from pinecone import ServerlessSpec
index_name = "langchain-test-index"  # change if desired

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
# Connect to the index
index = pc.Index(index_name)

In [61]:
index

In [62]:
# Importing PineconeVectorStore from langchain_pinecone
from langchain_pinecone import PineconeVectorStore 
# Create a PineconeVectorStore instance
vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [63]:
# Step 6: Make the retriever runnable
retriever = vector_store.as_retriever()


In [64]:
results=vector_store.similarity_search("what is Agentic AI")

In [65]:
results

[Document(id='52cf7ae6-2c42-47c5-9bc8-e5470db9f418', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='1248bbf5-b351-4197-b2b1-0ee5b7c987eb', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='5b46701a-9055-44b0-b4e5-d472751e276e', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='0732cd2a-7960-4991-806d-2b375b472c44', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :(')]

In [66]:
from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
uuids = [str(uuid4()) for _ in range(len(documents))]
vector_store.add_documents(documents=documents, ids=uuids)

['606bc3b8-3af3-4ac8-be52-931fdacde766',
 '9f9c5ec3-40d7-4b87-99b4-05ece8f9cd58',
 'ef70b37f-c9e1-442d-bb74-d68218c5e50d',
 '2fa76c71-4920-4385-aa21-1eae12a77324',
 '8beb5150-6c31-4efd-9c8a-bd1d02b3f5da',
 '8b7dc11e-29a9-48ae-ad14-9e97a5c13381',
 'bd39998d-13da-44b7-a834-0de0b91b82f7',
 '5c4a8957-081c-4ad2-b189-a2683234f1d5',
 'b3cb75de-9bdb-4ac9-a174-e066003d1f39',
 '4d35d6c2-23f8-43c7-9a98-e135b9a4cb01']

In [67]:
retriever

VectorStoreRetriever(tags=['PineconeVectorStore', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x000001A9B0B70E00>, search_kwargs={})

In [68]:
results=vector_store.similarity_search("what is Agentic AI",filter={"source": "tweet"})
results

[Document(id='1248bbf5-b351-4197-b2b1-0ee5b7c987eb', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='52cf7ae6-2c42-47c5-9bc8-e5470db9f418', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='5b46701a-9055-44b0-b4e5-d472751e276e', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='0732cd2a-7960-4991-806d-2b375b472c44', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :(')]

In [69]:
vector_store.delete(ids=[uuids[-1]])

In [70]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.0)

In [71]:
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

In [72]:
import pprint

pprint.pprint(prompt.messages)

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


In [73]:
from langchain_core.output_parsers import StrOutputParser # Imports the StrOutputParser class from the langchain_core.output_parsers module. This class is used to parse the output of the LLM into a string format.
from langchain_core.runnables import RunnablePassthrough # RunnablePassthrough is a class that allows you to pass through the input without any modifications.

In [74]:
def format_docs(docs): # This function formats the documents by joining their page content with two newlines.
    return "\n\n".join(doc.page_content for doc in docs)

In [75]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Wrap format_docs into a Runnable
format_docs_runnable = RunnableLambda(format_docs)
# Create the RAG chain
format_docs_runnable

RunnableLambda(format_docs)

In [76]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [78]:
rag_chain.invoke("langchain ?")

"LangChain is a framework used for building applications with large language models.  The provided text highlights its use in a new project.  More information about LangChain's capabilities is needed for a more complete answer."